In [ ]:
# import subprocess
# import os

# # Define the base directory where you want to save the files
# base_dir = "sam_dictionaries"

# # Ensure the base directory exists
# os.makedirs(base_dir, exist_ok=True)

# # List of URLs to download
# directories = [f"https://baulab.us/u/smarks/autoencoders/pythia-70m-deduped/attn_out_layer{layer}/10_32768/ae.pt" for layer in range(6)]

# for url in directories:
#     # Extract the layer number from the URL for directory naming
#     layer = url.split("attn_out_layer")[1].split("/")[0]
#     # Define the local directory structure
#     local_dir = os.path.join(base_dir, f"attn_out_layer{layer}", "10_32768")
#     # Ensure the local directory structure exists
#     os.makedirs(local_dir, exist_ok=True)
#     # Run wget command, specifying the current directory to save the file
#     subprocess.run(["wget", "-P", local_dir, "--no-parent", url], check=True)


In [1]:
import torch
from nnsight import LanguageModel
from sam_dictionary import AutoEncoder

ae_list = []

for layer in range(6):
    path = f"./sam_dictionaries/attn_out_layer{layer}/10_32768/ae.pt"

    activation_dim = 512 # dimension of the NN's activations to be autoencoded
    dictionary_size = 64 * activation_dim # number of features in the dictionary
    ae = AutoEncoder(activation_dim, dictionary_size)
    ae.load_state_dict(torch.load(path))
    ae.to("cuda:0")

    ae_list.append(ae)


print("Loaded dictionaries")
        
pythia = LanguageModel("EleutherAI/pythia-70m", device_map="cuda:0", dispatch=True)

print("Loaded GPT")

kwargs = {
    "load_in_4bit": True,
    "device_map": "cuda:0",
    "dispatch": True
}

mixtral = LanguageModel("mistralai/Mixtral-8x7B-Instruct-v0.1", **kwargs)

print("Loaded Mixtral in 4bit")

Loaded dictionaries


config.json:   0%|          | 0.00/567 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/396 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/166M [00:00<?, ?B/s]

Loaded GPT


The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


Loading checkpoint shards:   0%|          | 0/19 [00:00<?, ?it/s]

Loaded Mixtral in 4bit


In [16]:
messages = [
    {
        "role": "user", 
        "content": "For the rest of the conversation, return your answer in brackets, e.g. {ANSWER}. Just return an answer, nothing else. What is the capital of France?"
    },
    {
        "role": "assistant", 
        "content": "{Paris}"
    },
    {
        "role": "user", 
        "content": "What is a similar sentence to the following: 'I like pizza and big-ass big-scale ham.'? Return one sentence in brackets, nothing else."
    },
]

tok = mixtral.tokenizer.apply_chat_template(messages, return_tensors="pt")

In [17]:
with mixtral.generate(tok, scan=False, validate=False, max_new_tokens=50):
    out = mixtral.generator.output.save()

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


In [18]:
print(mixtral.tokenizer.decode(out[0]))

<s> [INST] For the rest of the conversation, return your answer in brackets, e.g. {ANSWER}. Just return an answer, nothing else. What is the capital of France? [/INST]{Paris}</s> [INST] What is a similar sentence to the following: 'I like pizza and big-ass big-scale ham.'? Return one sentence in brackets, nothing else. [/INST]{'I enjoy pizza and large-scale, generously proportioned ham.'}</s>


In [ ]:
with pythia.trace("trace"):
    activation = pythia.gpt_neox.layers[0].attention.output[0].save()

    reconstructed = ae_list[0].encoder(activation)

    reconstructed.save()

In [ ]:
from simulator import Agent, RewardEngine, Location, Feature

reward_engine = RewardEngine(mixtral, ae_list)

agent = Agent(mixtral, reward_engine)

location = Location("attn", 0, 34545)

feature = Feature(
    tokens = ["hello", "world"],
    acts = [0, 1, 2, 3, 4, 5]
    n_acts = [0, 1, 2, 3, 4, 5]
    location  = location
    )

explaination = agent(feature)